# Đánh giá mô hình **Qwen3-4B** trên bộ dữ liệu Số phức (5022 câu: 66 gốc + 4956 nhân bản) — Kaggle 2×GPU

Giải toàn bộ câu trắc nghiệm bằng **Qwen3-4B** (baseline chính, khớp bản Ollama `qwen3:4b` cũ), dùng **đúng prompt** đã dùng để gọi Gemini trong `solve_nhan_ban_with_models.py` — để so sánh công bằng giữa các model. Trích đáp án A/B/C/D, so với đáp án đúng, tính accuracy (tổng thể + theo vấn đề / mức độ / loại số / dạng), đóng gói kết quả thành **1 file .zip** để tải về (dùng được cả khi chạy *Save & Run All*).

## Bật/tắt suy luận (thinking) — 1 công tắc
Ở cell CẤU HÌNH, đổi `ENABLE_THINKING` là đủ (token budget tự điều chỉnh theo):
- **`False`:** trả lời thẳng, ~3072 token/câu → full 5022 câu ~6.5-7h, VẪN nên chia sharding nếu muốn dưới 12h an toàn.
- **`True` (chất lượng cao hơn, đã đo 89.39% trên 66 câu gốc, Sai=0):** budget 9216 token/câu → full 5022 câu ước **~20-21h, VƯỢT XA 12h nếu chạy 1 lần** → BẮT BUỘC dùng **SHARDING** bên dưới.

## SHARDING — chạy 4 Kaggle SONG SONG cùng lúc
Bạn có 4 tài khoản/phiên Kaggle → chạy **song song, không phải tuần tự**: mỗi Kaggle nhận 1/4 dữ liệu, cả 4 chạy đồng thời, xong cùng lúc (~5-5.5h với think-mode) thay vì chạy nối tiếp 1 tài khoản nhiều lần.

**Việc cần chỉnh ở MỖI Kaggle (cell CẤU HÌNH bên dưới) — CHỈ 1 dòng khác nhau:**
- Giữ `NUM_SHARDS = 4` giống nhau ở cả 4.
- Kaggle #1: `SHARD_INDEX = 0`
- Kaggle #2: `SHARD_INDEX = 1`
- Kaggle #3: `SHARD_INDEX = 2`
- Kaggle #4: `SHARD_INDEX = 3`
- Mọi thứ khác (MODEL, ENABLE_THINKING, PARALLEL_MODE...) giữ NGUYÊN giống nhau ở cả 4 Kaggle.

Dữ liệu được chia XEN KẼ (`records[SHARD_INDEX::NUM_SHARDS]`) nên độ khó/dạng câu cân bằng đều giữa 4 ca — mỗi ca nhận đúng 1/4 tổng số câu (~1255-1256 câu). Sau khi cả 4 chạy xong, tải 4 file zip về, giải nén, rồi chạy `scratch/merge_shards.py` (đã cập nhật cho 4 shard) để ghép thành 1 bộ kết quả đầy đủ 5022 câu.

Nếu chỉ muốn test nhanh/không có đủ 4 Kaggle: đặt `NUM_SHARDS = 1` để chạy hết trong 1 lần (phù hợp cho no-think, hoặc dùng `MAX_QUESTIONS` để lấy mẫu nhỏ).

## Về tốc độ (đã đo & sửa cách tính)
Thời gian nạp model chỉ 1 lần (~2-5 phút) được bấm giờ RIÊNG, KHÔNG tính vào tốc độ/câu. Đã thử 8B: chỉ hơn 4B ~1 câu (75% vs 70%), chậm hơn ~30%, vẫn 0% ở nhóm cực trị → **không đáng**, dùng 4B.

## Chuẩn bị (Settings bên phải) — làm GIỐNG NHAU ở cả 4 Kaggle
1. **Accelerator → GPU T4 ×2**.
2. **Internet → On** (tải model từ Hugging Face).
3. **Add Input**: `So_phuc_day_du.xlsx` **hoặc** `nhan_ban.json` (+ tùy chọn `cau_goc_data.json`).
4. **Save Version → Save & Run All (Commit)** — làm ở cả 4 phiên cùng lúc, chỉ khác `SHARD_INDEX`.

## Dùng 2 GPU: tensor-parallel
Đã đo thực tế trên Kaggle: với model 4B, `tensor_parallel_size=2` **nhanh hơn** cách chia dữ liệu chạy 2 engine độc lập (mỗi engine chỉ giữ 1 nửa trọng số → còn dư nhiều bộ nhớ cho KV-cache/độ song song hơn, thắng cả chi phí giao tiếp giữa 2 GPU không NVLink). Notebook dùng continuous batching của vLLM (đưa toàn bộ prompt vào 1 lần `generate`) — sinh 1 lượt duy nhất, không có cơ chế "ép" gì cả.


In [ ]:
# Cài vLLM MỚI (Qwen3 cần vLLM đủ mới). Mất ~3-5 phút.
!pip install -q -U vllm
# Gỡ torchcodec: bản mới nhất cố nạp libnvrtc.so.13 (CUDA 13) trong khi Kaggle chỉ có
# CUDA 12.x -> sập lúc import vllm (RuntimeError, không phải ImportError nên vllm không
# tự fallback được). Ta chỉ dùng text-only nên không cần torchcodec (chỉ phục vụ video multimodal).
# Gỡ xong thì "from torchcodec... import ImportError" trong vllm sẽ bắt được và dùng placeholder.
!pip uninstall -y -q torchcodec
import vllm, transformers
print("vLLM:", vllm.__version__, "| transformers:", transformers.__version__)

In [ ]:
import os, re, json, glob, time, random
import pandas as pd

# ===================== CẤU HÌNH =====================
MODEL = "Qwen/Qwen3-4B"       # baseline chính (khớp bản Ollama qwen3:4b cũ). 8B không đáng đổi.

# >>> CÔNG TẮC DUY NHẤT để bật/tắt suy luận - token budget TỰ điều chỉnh theo bên dưới <<<
ENABLE_THINKING = True        # True  = bật suy luận Qwen3 (đã đo trên 66 câu gốc: 89.39% đúng, Sai=0,
                              #         so với 84.85% của no-think). Full run BẮT BUỘC dùng NUM_SHARDS=4.
                              # False = trả lời thẳng (~4-4.5h full run, 1 commit là xong, kém hơn ~4-5%).

PARALLEL_MODE  = "tensor"     # "tensor" (khuyên dùng, đã đo nhanh hơn) hoặc "data" (dự phòng)
TENSOR_PARALLEL = 2           # chỉ dùng khi PARALLEL_MODE="tensor"
GPU_MEM_UTIL    = 0.90
SEED            = 42          # để tái lập kết quả
ONLY_CAU_GOC    = False        # True = CHỈ chạy 66 câu gốc (bỏ qua 4956 câu nhân bản). False = chạy toàn bộ dataset.
MAX_QUESTIONS   = None         # None = chạy hết. Đặt số (vd 30) để lấy mẫu ngẫu nhiên (áp dụng SAU lọc ONLY_CAU_GOC).

# >>> SHARDING - chia dữ liệu cho 4 KAGGLE CHẠY SONG SONG (không phải tuần tự) <<<
# Think-mode full 5022 câu ước ~20-21h (VƯỢT XA 12h) -> chia 4 ca chạy ĐỒNG THỜI trên 4 tài
# khoản/phiên Kaggle, mỗi ca ~1255-1256 câu ~5-5.5h (an toàn, xong cùng lúc).
# Cách dùng: 4 Kaggle giữ NUM_SHARDS=4 giống nhau, CHỈ đổi SHARD_INDEX: 0 / 1 / 2 / 3.
# Chia XEN KẼ (records[SHARD_INDEX::NUM_SHARDS]) nên độ khó/dạng câu cân bằng đều giữa 4 ca.
# Sau khi cả 4 xong: tải 4 zip về, giải nén, chạy scratch/merge_shards.py để ghép thành 1 bộ 5022 câu.
# Đặt NUM_SHARDS=1 để chạy tất cả trong 1 lần (dùng cho no-think, hoặc khi test mẫu nhỏ).
NUM_SHARDS  = 4                # think-mode full, chạy song song 4 Kaggle: giữ 4 ở CẢ 4 phiên
SHARD_INDEX = 0                # Kaggle #1: 0 | #2: 1 | #3: 2 | #4: 3 (giữ nguyên mọi thứ khác)

OUT_DIR         = "/kaggle/working"

if ENABLE_THINKING:
    MAX_MODEL_LEN  = 10240
    MAX_NEW_TOKENS = 9216   # nâng từ 8000: phân tích 66 câu cho thấy 3/7 câu "Không xác định" đã suy luận
                            # XONG nhưng bị cắt giữa lúc viết lời giải cuối -> thêm ~1200 token cứu được nhóm này.
                            # (4/7 còn lại loop vô tận, tăng bao nhiêu cũng không cứu - đừng tăng thêm nữa.)
    TEMPERATURE, TOP_P, TOP_K = 0.6, 0.95, 20   # Qwen3 khuyến nghị cho chế độ THINKING
else:
    MAX_MODEL_LEN  = 4096
    MAX_NEW_TOKENS = 3072                         # đã tăng từ 1536 -> 3072 (công bằng hơn với Gemini)
    TEMPERATURE, TOP_P, TOP_K = 0.7, 0.8, 20     # Qwen3 khuyến nghị cho chế độ KHÔNG thinking
# ===================================================
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print(f"Model: {MODEL} | parallel_mode: {PARALLEL_MODE} | thinking: {ENABLE_THINKING} "
      f"| max_new_tokens: {MAX_NEW_TOKENS} | max_model_len: {MAX_MODEL_LEN}")

In [ ]:
import torch
N_GPU = torch.cuda.device_count()
print("Số GPU nhìn thấy:", N_GPU)
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} - {p.total_memory/1024**3:.1f} GB")

if PARALLEL_MODE == "data":
    n_shards = min(2, N_GPU)
    print(f"=> Sẽ chạy DATA-PARALLEL trên {n_shards} GPU độc lập (mỗi GPU 1 engine riêng, tensor_parallel_size=1).")
elif PARALLEL_MODE == "tensor":
    if N_GPU < TENSOR_PARALLEL:
        print(f"CẢNH BÁO: chỉ có {N_GPU} GPU < TENSOR_PARALLEL={TENSOR_PARALLEL}. Hạ xuống {max(1,N_GPU)}.")
        TENSOR_PARALLEL = max(1, N_GPU)
    print(f"=> Sẽ chạy TENSOR-PARALLEL với tensor_parallel_size={TENSOR_PARALLEL}.")

In [ ]:
def find_file(names):
    for nm in names:
        hits = glob.glob(f"/kaggle/input/**/{nm}", recursive=True)
        if hits:
            return sorted(hits)[0]
    return None

def parse_correct(ans):
    m = re.match(r"\s*([ABCD])", str(ans))
    return m.group(1) if m else ""

records = []
xlsx    = find_file(["So_phuc_day_du.xlsx"])
nb_json = find_file(["nhan_ban.json"])

if xlsx:
    print("Đọc dữ liệu từ Excel:", xlsx)
    df = pd.read_excel(xlsx)
    for _, r in df.iterrows():
        records.append({
            "stt": r.get("STT"), "stt_goc": r.get("STT câu gốc"),
            "van_de": r.get("Vấn đề"), "ma_dang": r.get("Mã dạng"),
            "muc_do": r.get("Mức độ"), "ten_dang": r.get("Tên dạng"),
            "nguon": r.get("Nguồn"), "loai_so": r.get("Loại số"),
            "de_bai": str(r.get("Đề bài")),
            "correct": parse_correct(r.get("Đáp án đúng")),
        })
elif nb_json:
    print("Đọc dữ liệu từ JSON:", nb_json)
    for r in json.load(open(nb_json, encoding="utf-8")):
        records.append({
            "stt": None, "stt_goc": r.get("stt_goc"),
            "van_de": r.get("van_de"), "ma_dang": r.get("ma_dang"),
            "muc_do": r.get("muc_do"), "ten_dang": r.get("ten_dang"),
            "nguon": "Nhân bản", "loai_so": r.get("loai_so"),
            "de_bai": str(r.get("de_bai")),
            "correct": parse_correct(r.get("dap_an")),
        })
    goc_json = find_file(["cau_goc_data.json"])
    if goc_json:
        for r in json.load(open(goc_json, encoding="utf-8")):
            records.append({
                "stt": r.get("stt"), "stt_goc": r.get("stt"),
                "van_de": None, "ma_dang": None, "muc_do": None,
                "ten_dang": None, "nguon": "Câu gốc", "loai_so": None,
                "de_bai": str(r.get("deBai")),
                "correct": parse_correct(r.get("dapAn")),
            })
else:
    raise FileNotFoundError(
        "Không thấy So_phuc_day_du.xlsx hoặc nhan_ban.json trong /kaggle/input. "
        "Hãy Add Input file dữ liệu."
    )

records = [r for r in records if r["correct"] in "ABCD"
           and r["de_bai"] and r["de_bai"] != "nan"]

if ONLY_CAU_GOC:
    records = [r for r in records if r["nguon"] == "Câu gốc"]
    print(f"(ONLY_CAU_GOC=True: chỉ giữ lại {len(records)} câu gốc, bỏ qua toàn bộ câu nhân bản)")

if MAX_QUESTIONS:
    # Lấy MẪU NGẪU NHIÊN (không phải N dòng đầu) để mẫu thử đại diện đủ cả
    # "Câu gốc" lẫn "Nhân bản" (khi ONLY_CAU_GOC=False), đủ các mức độ/vấn đề khác nhau.
    records = random.Random(123).sample(records, min(MAX_QUESTIONS, len(records)))
    print(f"(chế độ CHẠY THỬ: lấy mẫu ngẫu nhiên {len(records)} câu)")

if NUM_SHARDS > 1:
    # Chia XEN KẼ để cân bằng độ khó/dạng câu giữa các ca chạy.
    records = records[SHARD_INDEX::NUM_SHARDS]
    print(f"(SHARDING: ca {SHARD_INDEX + 1}/{NUM_SHARDS} -> nhận {len(records)} câu)")

print("Tổng số câu sẽ giải:", len(records))

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)

# Prompt GIỐNG HỆT bản dùng để gọi Gemini trong solve_nhan_ban_with_models.py (repo gốc)
# - không có gì chuyên biệt cho Qwen - để so sánh công bằng giữa các model.
PROMPT_TEMPLATE = """Bạn hãy giải bài toán trắc nghiệm sau đây.

Yêu cầu trình bày:
- Giải chi tiết từng bước bằng tiếng Việt.
- Viết dưới dạng Markdown, dùng LaTeX cho công thức toán (đặt công thức trong dấu $...$).
- Không xuống dòng quá nhiều lần trong một biểu thức (ví dụ ngoặc vuông, phân số) vì sẽ khó đọc.
- Sau khi giải xong, kết luận rõ ràng ở dòng cuối cùng: "Đáp án đúng: X" (X là A, B, C hoặc D).

Đề bài:
{de_bai}
"""

def make_prompt(de_bai):
    text = PROMPT_TEMPLATE.format(de_bai=de_bai.strip())
    # Không thêm system message riêng - chỉ 1 lượt user, giống cách gọi Gemini (1 đoạn văn bản thuần).
    msgs = [{"role": "user", "content": text}]
    return tok.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True,
        enable_thinking=ENABLE_THINKING,   # bật/tắt suy luận Qwen3 theo cấu hình
    )

prompts = [make_prompt(r["de_bai"]) for r in records]
print("Số prompt:", len(prompts))

# Chẩn đoán độ dài prompt để chọn MAX_MODEL_LEN hợp lý
sample_check = prompts if len(prompts) <= 500 else random.Random(7).sample(prompts, 500)
plen = [len(tok(p)["input_ids"]) for p in sample_check]
print(f"Độ dài prompt (token): min={min(plen)} max={max(plen)} trung_bình={sum(plen)/len(plen):.0f}")
budget = max(plen) + MAX_NEW_TOKENS + 64
print(f"Ngưỡng cần thiết ước tính: ~{budget} token | MAX_MODEL_LEN hiện tại: {MAX_MODEL_LEN}")
if budget > MAX_MODEL_LEN:
    print("CẢNH BÁO: MAX_MODEL_LEN có thể chưa đủ cho câu prompt dài nhất — cân nhắc tăng lên.")

print("----- Ví dụ prompt -----")
print(prompts[0][:800], "...")

In [ ]:
%%writefile /kaggle/working/dp_worker.py
"""Worker: chạy đúng 1 engine vLLM trên đúng 1 GPU (dùng cho data-parallel qua subprocess)."""
import argparse
import json
import os


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--gpu", type=int, required=True)
    ap.add_argument("--model", required=True)
    ap.add_argument("--prompts_file", required=True)
    ap.add_argument("--out_file", required=True)
    ap.add_argument("--max_model_len", type=int, default=3072)
    ap.add_argument("--gpu_mem_util", type=float, default=0.90)
    ap.add_argument("--max_new_tokens", type=int, default=1536)
    ap.add_argument("--temperature", type=float, default=0.7)
    ap.add_argument("--top_p", type=float, default=0.8)
    ap.add_argument("--top_k", type=int, default=20)
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()

    os.environ["CUDA_VISIBLE_DEVICES"] = str(args.gpu)
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

    with open(args.prompts_file, encoding="utf-8") as f:
        prompts = json.load(f)

    if not prompts:
        with open(args.out_file, "w", encoding="utf-8") as f:
            json.dump([], f)
        print(f"[GPU {args.gpu}] Không có câu nào được giao, bỏ qua.")
        return

    from vllm import LLM, SamplingParams

    llm = LLM(
        model=args.model,
        tensor_parallel_size=1,     # chỉ thấy đúng 1 GPU (đã set qua CUDA_VISIBLE_DEVICES)
        dtype="float16",
        max_model_len=args.max_model_len,
        gpu_memory_utilization=args.gpu_mem_util,
        trust_remote_code=True,
        enforce_eager=True,  # tránh OOM lúc capture CUDA graph FULL trên T4 (xem cell tensor-parallel)
        seed=args.seed,
    )
    sampling = SamplingParams(
        temperature=args.temperature, top_p=args.top_p, top_k=args.top_k,
        max_tokens=args.max_new_tokens, seed=args.seed,
    )
    outputs = llm.generate(prompts, sampling)
    texts = [o.outputs[0].text for o in outputs]
    with open(args.out_file, "w", encoding="utf-8") as f:
        json.dump(texts, f, ensure_ascii=False)
    print(f"[GPU {args.gpu}] Xong {len(texts)} câu -> {args.out_file}")


if __name__ == "__main__":
    main()

In [ ]:
import subprocess, sys

WORKER_PATH = "/kaggle/working/dp_worker.py"


def run_data_parallel():
    """2 tiến trình vLLM độc lập, mỗi tiến trình 1 GPU (CUDA_VISIBLE_DEVICES riêng)."""
    n_shards = min(2, N_GPU)
    print(f"Data-parallel: {n_shards} GPU độc lập, mỗi GPU 1 engine riêng (tensor_parallel_size=1).")

    shard_idx = [[] for _ in range(n_shards)]
    for i in range(len(prompts)):
        shard_idx[i % n_shards].append(i)   # xen kẽ để cân bằng độ dài giữa 2 GPU

    procs = []
    for s in range(n_shards):
        idxs = shard_idx[s]
        pf = f"/kaggle/working/dp_prompts_{s}.json"
        of = f"/kaggle/working/dp_out_{s}.json"
        with open(pf, "w", encoding="utf-8") as f:
            json.dump([prompts[i] for i in idxs], f, ensure_ascii=False)
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(s))
        cmd = [sys.executable, WORKER_PATH,
               "--gpu", str(s), "--model", MODEL,
               "--prompts_file", pf, "--out_file", of,
               "--max_model_len", str(MAX_MODEL_LEN),
               "--gpu_mem_util", str(GPU_MEM_UTIL),
               "--max_new_tokens", str(MAX_NEW_TOKENS),
               "--temperature", str(TEMPERATURE), "--top_p", str(TOP_P),
               "--top_k", str(TOP_K), "--seed", str(SEED)]
        print(f"-> Khởi chạy GPU {s}: {len(idxs)} câu")
        procs.append((subprocess.Popen(cmd, env=env), of, idxs))

    texts_local = [None] * len(prompts)
    for proc, of, idxs in procs:
        ret = proc.wait()
        if ret != 0:
            raise RuntimeError(f"Worker GPU thất bại (exit code {ret}). Xem log ở trên để biết chi tiết.")
        with open(of, encoding="utf-8") as f:
            shard_out = json.load(f)
        for i, txt in zip(idxs, shard_out):
            texts_local[i] = txt
    return texts_local


def run_tensor_parallel():
    """1 engine, model chia trên nhiều GPU qua tensor-parallel (mặc định).
    Trả về (texts, gen_dt) - gen_dt CHỈ tính thời gian sinh, KHÔNG gồm nạp model."""
    from vllm import LLM, SamplingParams   # đã import ở đầu cell
    t_load = time.time()
    llm = LLM(
        model=MODEL, tensor_parallel_size=TENSOR_PARALLEL, dtype="float16",
        max_model_len=MAX_MODEL_LEN, gpu_memory_utilization=GPU_MEM_UTIL,
        trust_remote_code=True, enforce_eager=True, seed=SEED,
        # enforce_eager=True: tắt CUDA graph capture. vLLM mới mặc định capture cả
        # graph FULL (không chỉ PIECEWISE), tốn thêm VRAM ngoài phần gpu_memory_utilization
        # danh cho KV-cache -> OOM trên T4 16GB khi TENSOR_PARALLEL=2. Chậm hơn ~10-20%
        # nhưng chạy được ổn định, không sập giữa chừng.
    )
    print(f"  (nạp model + khởi tạo engine: {(time.time()-t_load)/60:.1f} phút - CHỈ 1 lần, KHÔNG tính vào tốc độ/câu)")
    sampling = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
                              max_tokens=MAX_NEW_TOKENS, seed=SEED)
    t_gen = time.time()
    outputs = llm.generate(prompts, sampling)
    gen_dt = time.time() - t_gen
    return [o.outputs[0].text for o in outputs], gen_dt


if PARALLEL_MODE == "data":
    t0 = time.time()
    texts = run_data_parallel()
    gen_dt = time.time() - t0
    note = "(data mode: số này GỒM cả nạp model ở mỗi worker nên sẽ hơi cao)"
else:
    texts, gen_dt = run_tensor_parallel()
    note = "(chỉ tính phần SINH, đã tách riêng thời gian nạp model)"

print(f"Sinh {len(texts)} câu mất {gen_dt/60:.1f} phút {note} -> ~{gen_dt/max(1,len(texts)):.2f}s/câu.")
if MAX_QUESTIONS:
    full = 5022
    print(f"=> Ước tính SINH cho {full} câu (cả 5022, chưa chia shard): ~{gen_dt/len(texts)*full/60:.0f} phút "
          f"(cộng thêm ~4-5 phút nạp model 1 lần; con số tham khảo, chạy full thường sát hoặc nhanh hơn "
          f"do batch được lấp đầy tốt hơn so với mẫu nhỏ). Với NUM_SHARDS=4, mỗi ca ~1/4 số đó.")

In [ ]:
# ===== Trích đáp án A/B/C/D (bản đã VÁ so với grade_nhan_ban_answers.py) =====
# 3 cải tiến so với bản cũ:
#   1. Thêm mẫu tiếng ANH ("answer is X", "correct answer is X") - Qwen đôi khi kết luận bằng tiếng Anh.
#   2. Lấy khớp GẦN CUỐI VĂN BẢN NHẤT trong TẤT CẢ mẫu (thay vì "mẫu nào khớp trước thì lấy") -
#      xử lý đúng khi model nói đáp án giữa chừng rồi tự sửa ở cuối.
#   3. BỎ cơ chế dự phòng "bắt chữ cái A-D rời rạc" (dễ nhầm tên điểm hình học A/B/C) -
#      câu không có dấu hiệu kết luận rõ ràng -> để "Không xác định" cho người soi tay (an toàn hơn tự chấm sai).
# Mọi mẫu đều yêu cầu có TỪ KHÓA đáp án ("đáp án"/"chọn"/"answer"/\boxed) ngay trước chữ cái.

STRONG_PATTERNS = [
    r"[Đđ]áp\s*án\s*(?:đúng|cuối\s*cùng)?\s*(?:là)?\s*[:\-]?\s*\**\s*\(?\s*([A-D])\b",
    r"[Cc]họn\s*(?:đáp\s*án|phương\s*án)?\s*[:\-]?\s*\**\s*\(?\s*([A-D])\b",
    r"(?:correct\s+)?answer\s*(?:is|:|=)?\s*\**\s*\(?\s*([A-D])\b",
    r"final\s+answer\s*(?:is|:|=)?\s*\**\s*\(?\s*([A-D])\b",
    r"\\boxed\{\s*(?:\\text\{)?\s*([A-D])\b",
]

def parse_choices(de_bai):
    ch = {}
    for line in str(de_bai).split("\n"):
        line = line.strip()
        if len(line) > 2 and line[0] in "ABCD" and line[1] == ".":
            ch[line[0]] = line[2:].strip()
    return ch

def normalize(s):
    s = str(s).strip().rstrip(".").strip().replace("$", "").replace(" ", "")
    return re.sub(r"\\text\{([^}]*)\}", r"\1", s)

def extract_choice(text, choices=None):
    if not text or not str(text).strip():
        return ""
    text = str(text)
    # Với chế độ thinking: chỉ xét phần SAU </think> (câu trả lời thật).
    # Nếu bị cắt giữa lúc suy luận (có <think> mở nhưng chưa đóng </think>) -> chưa có kết luận.
    if "</think>" in text:
        body = text.split("</think>")[-1]
    elif "<think>" in text:
        body = ""
    else:
        body = text
    # 1. Lấy khớp gần CUỐI nhất trong tất cả mẫu mạnh (VN + EN + boxed)
    best_pos, best_letter = -1, ""
    for pat in STRONG_PATTERNS:
        for m in re.finditer(pat, body, flags=re.IGNORECASE):
            if m.end() > best_pos:
                best_pos, best_letter = m.end(), m.group(1).upper()
    if best_letter:
        return best_letter
    # 2. So khớp nội dung \boxed{...} với nội dung từng phương án A/B/C/D
    if choices:
        boxed = re.findall(r"\\boxed\{([^{}]*)\}", body)
        if boxed:
            val = normalize(boxed[-1])
            for L, c in choices.items():
                if val and normalize(c) == val:
                    return L
    # 3. Không có dấu hiệu kết luận rõ ràng -> để trống (Không xác định) cho người soi tay.
    return ""

rows = []
meta_keys = ["stt", "stt_goc", "van_de", "ma_dang", "muc_do", "ten_dang", "nguon", "loai_so"]
for r, txt in zip(records, texts):
    pred = extract_choice(txt, parse_choices(r["de_bai"]))
    status = "Không xác định" if pred == "" else ("Đúng" if pred == r["correct"] else "Sai")
    row = {k: r[k] for k in meta_keys}
    row.update({"dap_an_dung": r["correct"], "qwen_chon": pred,
                "ket_qua": status, "loi_giai_qwen": txt})
    rows.append(row)

res = pd.DataFrame(rows)
print("Đã chấm xong", len(res), "câu.")
print("(Lưu ý: có thể chấm LẠI file CSV này ở local bằng extractor cải tiến hơn mà KHÔNG cần chạy lại model.)")

In [ ]:
n = len(res)
n_dung   = int((res["ket_qua"] == "Đúng").sum())
n_sai    = int((res["ket_qua"] == "Sai").sum())
n_unknow = int((res["ket_qua"] == "Không xác định").sum())

lines = []
def out(s=""):
    print(s); lines.append(s)

out("=" * 55)
out(f"MODEL: {MODEL} | parallel_mode={PARALLEL_MODE} | thinking={ENABLE_THINKING}")
out(f"Thời gian sinh : {gen_dt/60:.1f} phút | {gen_dt/max(1,n):.2f} giây/câu (không tính nạp model)")
out(f"Tổng số câu    : {n}")
out(f"Đúng           : {n_dung}  ({n_dung/n*100:.2f}%)")
out(f"Sai            : {n_sai}  ({n_sai/n*100:.2f}%)")
out(f"Không xác định : {n_unknow}  ({n_unknow/n*100:.2f}%)  <- nếu lớn: tăng MAX_NEW_TOKENS")
out("=" * 55)

def acc_by(col):
    sub = res[res[col].notna()]
    if sub.empty:
        return None
    acc = sub.groupby(col)["ket_qua"].apply(lambda s: (s == "Đúng").mean() * 100).round(2)
    cnt = sub.groupby(col)["ket_qua"].size()
    return pd.DataFrame({"So_cau": cnt, "Accuracy_%": acc})

for col in ["van_de", "muc_do", "loai_so", "ma_dang", "nguon"]:
    tb = acc_by(col)
    if tb is not None:
        out(f"\n==== Accuracy theo {col} ====")
        out(tb.to_string())

# ---- lưu file ----
tag = MODEL.split("/")[-1].replace(".", "_") + ("_think" if ENABLE_THINKING else "_nothink")
if NUM_SHARDS > 1:
    tag += f"_shard{SHARD_INDEX + 1}of{NUM_SHARDS}"
csv_path  = os.path.join(OUT_DIR, f"ketqua_{tag}.csv")
xlsx_path = os.path.join(OUT_DIR, f"ketqua_{tag}.xlsx")
pred_path = os.path.join(OUT_DIR, f"predictions_{tag}.json")
sum_path  = os.path.join(OUT_DIR, f"summary_{tag}.txt")

res.to_csv(csv_path, index=False, encoding="utf-8-sig")
try:
    res.to_excel(xlsx_path, index=False)
except Exception as e:
    out(f"(Bỏ qua xuất Excel: {e})")

pred = res[["stt", "stt_goc", "dap_an_dung", "qwen_chon", "ket_qua"]].to_dict("records")
with open(pred_path, "w", encoding="utf-8") as f:
    json.dump({"model": MODEL, "parallel_mode": PARALLEL_MODE, "thinking": ENABLE_THINKING,
               "num_shards": NUM_SHARDS, "shard_index": SHARD_INDEX,
               "gen_seconds": round(gen_dt, 1), "gen_seconds_per_cau": round(gen_dt / max(1, n), 3),
               "accuracy_percent": round(n_dung / n * 100, 2),
               "n": n, "n_dung": n_dung, "n_sai": n_sai, "n_khong_xac_dinh": n_unknow,
               "predictions": pred}, f, ensure_ascii=False, indent=1)
with open(sum_path, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print("\nĐã lưu:", csv_path, "|", xlsx_path, "|", pred_path, "|", sum_path)

In [ ]:
# Đóng gói mọi thứ cần tải về thành 1 file .zip (tiện cho Save & Run All)
import zipfile
tag = MODEL.split("/")[-1].replace(".", "_") + ("_think" if ENABLE_THINKING else "_nothink")
if NUM_SHARDS > 1:
    tag += f"_shard{SHARD_INDEX + 1}of{NUM_SHARDS}"
zip_path = os.path.join(OUT_DIR, f"ketqua_{tag}.zip")
want = [f"ketqua_{tag}.csv", f"ketqua_{tag}.xlsx",
        f"predictions_{tag}.json", f"summary_{tag}.txt"]
files = [os.path.join(OUT_DIR, w) for w in want if os.path.exists(os.path.join(OUT_DIR, w))]
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for fp in files:
        z.write(fp, arcname=os.path.basename(fp))
print("Đã tạo ZIP:", zip_path, f"({os.path.getsize(zip_path)/1024:.0f} KB)")
for fp in files:
    print("  +", os.path.basename(fp))
print("\nTải về ở tab 'Output' (hoặc 'Data' của Version) sau khi commit xong.")

## Ghi chú
- **Chạy think-mode full = 4 Kaggle chạy SONG SONG** (xem hướng dẫn đầu notebook): `NUM_SHARDS=4` giống nhau ở cả 4 phiên, chỉ đổi `SHARD_INDEX` = 0/1/2/3. Mỗi ca ~1255-1256 câu, ~5-5.5h, an toàn dưới 12h, cả 4 xong gần như cùng lúc. Ghép 4 CSV bằng `scratch/merge_shards.py`.
- **Về nhóm "Không xác định" (phân tích 7 câu thất bại của lần chạy 66 câu)**: 3/7 câu đã suy luận xong (`</think>` đã đóng) nhưng bị cắt giữa lúc viết lời giải → budget 9216 (đã nâng từ 8000) nhắm cứu nhóm này. 4/7 còn lại loop vô tận trong lúc suy luận — tăng token bao nhiêu cũng không cứu, đừng nâng thêm nữa. `Sai=0` ở think-mode: model không chọn nhầm, chỉ không giải xong. Dữ liệu tham chiếu: KHÔNG câu đúng nào cần quá ~6927 token (đo thực tế, cap cũ 8000).
- **Chấm điểm tách rời khỏi chạy model**: notebook lưu NGUYÊN VĂN lời giải (`loi_giai_qwen`) trong CSV → có thể chấm lại ở local bằng extractor cải tiến mà KHÔNG cần chạy lại GPU. Cái duy nhất không lấy lại được là ngân sách token lúc sinh.
- **Extractor đã vá**: thêm mẫu tiếng Anh; lấy khớp gần cuối văn bản nhất; bỏ cơ chế "bắt chữ cái rời rạc" dễ nhầm tên điểm hình học. Với thinking: chỉ tính phần sau `</think>`.
- **Nên báo cáo minh bạch cả 3 nhóm** Đúng / Sai / Không xác định (đừng gộp "Không xác định" vào "Sai"). Dùng **cùng một extractor** khi chấm Gemini để so sánh công bằng.
- **PARALLEL_MODE = "tensor"** (đã đo nhanh hơn `"data"` với 4B trên T4 không NVLink). Nếu OOM: giảm `GPU_MEM_UTIL` (vd 0.85) hoặc `MAX_MODEL_LEN`.
- **Tùy chọn tăng tốc thêm (chưa kiểm chứng trên T4)**: đổi `MODEL = "Qwen/Qwen3-4B-AWQ"` (bản 4-bit chính thức của Qwen) — trọng số nhỏ hơn 4 lần → decode có thể nhanh hơn đáng kể và KV-cache dư hơn; về bản chất còn GẦN bản Ollama `qwen3:4b` gốc hơn (Ollama mặc định cũng là 4-bit Q4_K_M). NHƯNG kernel AWQ trên T4 với batch lớn chưa chắc nhanh hơn fp16 — nếu muốn dùng, PHẢI chạy A/B test 66 câu trước để so cả tốc độ lẫn accuracy rồi mới quyết.
- **Kết quả**: `ketqua_Qwen3-4B_<think|nothink>[_shardXofY].zip` trong Output gồm CSV (đầy đủ lời giải), XLSX, `predictions_*.json`, `summary_*.txt`.
